# Python Exceptions

---

## 1. Standard Exceptions (Most Common)

Python has a rich set of built-in exceptions. Here are the most frequently encountered:

| Exception | When it occurs |
|-----------|---------------|
| `NameError` | A variable or name is used before being defined |
| `ValueError` | Right type, but inappropriate value |
| `TypeError` | Operation applied to an object of the wrong type |
| `SyntaxError` | Code cannot be parsed by the interpreter |
| `OSError` | System-related failure (file not found, permission denied, etc.) |
| `RuntimeError` | Generic error that doesn't fit other categories |
| `Exception` | Base class for all non-system-exiting exceptions |

**Examples:**

```python
print(x)               # NameError: name 'x' is not defined
int("hello")           # ValueError: invalid literal for int()
"2" + 2                # TypeError: can only concatenate str (not "int") to str
open("missing.txt")    # FileNotFoundError (subclass of OSError)
```

---

## 2. Raising Exceptions

You can raise exceptions manually using the `raise` keyword.

```python
def divide(a, b):
    if b == 0:
        raise ValueError("Denominator cannot be zero.")
    return a / b

divide(10, 0)  # ValueError: Denominator cannot be zero.
```

You can also re-raise the current exception inside an `except` block:

```python
try:
    int("abc")
except ValueError:
    print("Caught it, re-raising...")
    raise  # re-raises the same ValueError
```

---

## 3. Handling Exceptions

Use `try-except` to catch and handle exceptions gracefully.

```python
try:
    result = 10 / 0
except ZeroDivisionError as e:
    print(f"Error: {e}")  # Error: division by zero
```

You can catch multiple exception types:

```python
try:
    value = int(input("Enter a number: "))
except (ValueError, TypeError) as e:
    print(f"Invalid input: {e}")
```

A bare `except Exception` catches almost everything (use with care):

```python
try:
    risky_operation()
except Exception as e:
    print(f"Something went wrong: {e}")
```

---

## 4. Exceptions Tree

Python exceptions follow a class hierarchy. Understanding it helps you choose the right catch level.

```
BaseException
├── SystemExit
├── KeyboardInterrupt
├── GeneratorExit
└── Exception
    ├── ArithmeticError
    │   ├── ZeroDivisionError
    │   └── OverflowError
    ├── LookupError
    │   ├── IndexError
    │   └── KeyError
    ├── OSError
    │   ├── FileNotFoundError
    │   └── PermissionError
    ├── TypeError
    ├── ValueError
    ├── NameError
    ├── RuntimeError
    └── StopIteration
```

> Catching a parent class (e.g., `LookupError`) will also catch its children (`IndexError`, `KeyError`).

---

## 5. User-Defined Exceptions

Create custom exceptions by subclassing `Exception` (or any of its subclasses).

```python
class InsufficientFundsError(Exception):
    def __init__(self, amount, balance):
        self.amount = amount
        self.balance = balance
        super().__init__(f"Cannot withdraw {amount}. Balance is only {balance}.")

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError(amount, balance)

try:
    withdraw(100, 250)
except InsufficientFundsError as e:
    print(e)  # Cannot withdraw 250. Balance is only 100.
```

---

## 6. `BaseException` vs `Exception`

| | `BaseException` | `Exception` |
|---|---|---|
| Root of | All exceptions | "Normal" exceptions only |
| Includes | `SystemExit`, `KeyboardInterrupt`, `GeneratorExit` | Everything under `Exception` |
| Use when | You need to catch truly everything | Standard error handling |

```python
# This silences Ctrl+C and sys.exit() — usually undesirable
try:
    ...
except BaseException:
    pass

# This is the correct, safe approach
try:
    ...
except Exception:
    pass
```

> **Rule of thumb:** Always inherit from `Exception` for custom exceptions, never `BaseException`, unless you have a very specific reason.

---

## 7. `try-except-else` and `try-finally`

### `else` — runs only if no exception was raised

```python
try:
    result = int("42")
except ValueError:
    print("Conversion failed.")
else:
    print(f"Success! Result: {result}")  # This runs
```

### `finally` — always runs, regardless of exception

```python
def read_file(path):
    f = open(path)
    try:
        return f.read()
    except OSError as e:
        print(f"Error: {e}")
    finally:
        f.close()  # Always executes, even on error or return
```

### Full pattern: `try-except-else-finally`

```python
try:
    value = int("10")
except ValueError:
    print("Bad input")
else:
    print(f"Parsed: {value}")   # runs if no exception
finally:
    print("Done.")              # always runs
```

---

## 8. Getting More Information: Tracebacks

The `traceback` built-in module lets you inspect, format, and log exception details programmatically.

```python
import traceback

try:
    1 / 0
except ZeroDivisionError:
    traceback.print_exc()           # Prints full traceback to stderr
    tb_str = traceback.format_exc() # Returns traceback as a string (useful for logging)
    print(tb_str)
```

You can also inspect the traceback object directly:

```python
import traceback, sys

try:
    int("bad")
except ValueError:
    exc_type, exc_value, exc_tb = sys.exc_info()
    frames = traceback.extract_tb(exc_tb)
    for frame in frames:
        print(f"File: {frame.filename}, Line: {frame.lineno}, In: {frame.name}")
```

---

## 9. Exception Chaining

Python supports explicit and implicit chaining to preserve context when one exception causes another.

### Implicit chaining (automatic)

```python
try:
    int("abc")
except ValueError:
    raise RuntimeError("Processing failed")
# RuntimeError: Processing failed
# During handling of the above exception, another exception occurred:
```

### Explicit chaining with `raise ... from`

```python
try:
    open("missing.txt")
except FileNotFoundError as e:
    raise RuntimeError("Config file missing") from e
```

### Suppress chaining with `raise ... from None`

```python
try:
    int("abc")
except ValueError:
    raise TypeError("Expected a number") from None  # Hides the original ValueError
```

---

## 10. `assert` Statement

`assert` is a debugging tool that raises `AssertionError` if a condition is `False`.

```python
def get_age(age):
    assert age >= 0, f"Age must be non-negative, got {age}"
    return age

get_age(-1)  # AssertionError: Age must be non-negative, got -1
```

You can catch it like any other exception:

```python
try:
    assert 1 == 2, "Math is broken"
except AssertionError as e:
    print(f"Assertion failed: {e}")
```

> **Warning:** `assert` statements are disabled when Python runs with the `-O` (optimize) flag. Do not use `assert` for data validation in production code — use explicit `if`/`raise` instead.